# Song Popularity Classification

## CMOR 438 Final Project: Music Analytics with a From-Scratch Machine Learning Package

This notebook presents a complete binary classification workflow for predicting relatively high-popularity songs from Spotify-style audio features using the custom `music_ml` package.

## 1) Problem Framing

We frame popularity prediction as a binary classification task:

- `target = 1` for songs with popularity at or above the 70th percentile
- `target = 0` otherwise

This setup focuses on identifying characteristics associated with relatively high-popularity tracks while keeping the model interpretable.

## 2) Load the Processed Spotify Dataset

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression as SklearnLR

# Make src package importable from notebooks/
sys.path.insert(0, str(Path("..").resolve() / "src"))

from music_ml.preprocessing import train_test_split, StandardScaler
from music_ml.supervised import LogisticRegression
from music_ml.metrics import accuracy_score, precision_score, recall_score, f1_score

data_path = Path("../data/processed/spotify_processed.csv")
df = pd.read_csv(data_path)

print(f"Loaded dataset from: {data_path}")
print(f"Shape: {df.shape}")
df.head()

## 3) Select Feature Columns

We use the following numeric audio features for classification.

In [ ]:
feature_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms",
]

required_cols = feature_cols + ["popularity"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_cols}")

modeling_df = df[required_cols].dropna().copy()
print(f"Rows after selecting required columns and dropping missing values: {len(modeling_df)}")
modeling_df.head()

## 4) Define Binary Target (`target`)

We define:

- `target = 1` if `popularity` is greater than or equal to the 70th percentile
- `target = 0` otherwise

In [ ]:
popularity_threshold = modeling_df["popularity"].quantile(0.70)
modeling_df["target"] = (modeling_df["popularity"] >= popularity_threshold).astype(int)

X = modeling_df[feature_cols].to_numpy()
y = modeling_df["target"].to_numpy()

print(f"70th percentile popularity threshold: {popularity_threshold:.3f}")
print(f"Feature matrix shape: {X.shape}")
print("Target distribution:")
print(pd.Series(y).value_counts().sort_index())

## 5) Train/Test Split

We use an 80/20 split to evaluate generalization on held-out data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=True,
    random_state=42,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

## 6) Standardize Features with Custom `StandardScaler`

Scaling is fitted on training data only, then applied to train and test splits.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")
print("Train means (approx. 0):", np.round(X_train_scaled.mean(axis=0), 3))

## 7) Train Custom `LogisticRegression`

The model learns a linear decision boundary in the standardized feature space.

In [ ]:
model = LogisticRegression(learning_rate=0.01, n_iters=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("Custom logistic regression trained.")
print(f"Final training loss: {model.loss_history_[-1]:.4f}")

## 8) Evaluate Classification Performance

We report four core binary-classification metrics: accuracy, precision, recall, and F1 score.

In [ ]:
results = pd.Series(
    {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    }
)

print("Custom LogisticRegression Performance")
for metric_name, metric_value in results.items():
    print(f"{metric_name:>10}: {metric_value:.4f}")

results.to_frame(name="score")

## 9) Feature Importance from Model Coefficients

For logistic regression, larger absolute coefficients indicate stronger influence on the prediction.
Positive values push predictions toward the high-popularity class (`target = 1`), while negative values push toward `target = 0`.

In [ ]:
coef_series = pd.Series(model.weights_, index=feature_cols).sort_values()

plt.figure(figsize=(8, 5))
coef_series.plot(kind="barh")
plt.title("Feature Importance from Logistic Regression Coefficients")
plt.xlabel("Coefficient Value (standardized feature space)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

coef_series

## 10) Interpretation and Practical Takeaways

Using the model outputs above (`coef_series`, `top_positive`, and `top_negative`), we can base interpretation on the learned weights rather than assumptions. In this dataset, the three largest positive coefficients are the features listed in `top_positive`, which means higher values on those features increase the model’s predicted odds of belonging to the high-popularity group (`target = 1`).

Likewise, the three most negative coefficients are the features listed in `top_negative`, indicating that higher values on those dimensions are associated with lower predicted odds of high popularity in this sample.

So, for this specific dataset, the model’s popularity profile is defined by the actual top positive and top negative weighted features reported above. These findings are dataset-dependent and should be interpreted as correlations learned from this training set, not universal rules about music popularity. These results should be interpreted as correlations within this dataset rather than causal relationships.

In [ ]:
top_positive = coef_series.sort_values(ascending=False).head(3)
top_negative = coef_series.sort_values(ascending=True).head(3)

print("Top positive features (more associated with high popularity):")
print(top_positive)
print("\nTop negative features (less associated with high popularity):")
print(top_negative)

## 11) Comparison with scikit-learn Logistic Regression

To benchmark the custom implementation, we train `SklearnLR` on the same standardized training data and evaluate with the same four metrics.

In [ ]:
sk_model = SklearnLR(max_iter=1000, random_state=42)
sk_model.fit(X_train_scaled, y_train)
y_pred_sk = sk_model.predict(X_test_scaled)

sk_results = pd.Series(
    {
        "accuracy": accuracy_score(y_test, y_pred_sk),
        "precision": precision_score(y_test, y_pred_sk),
        "recall": recall_score(y_test, y_pred_sk),
        "f1": f1_score(y_test, y_pred_sk),
    }
)

comparison_df = pd.DataFrame(
    {
        "custom_logistic": results,
        "sklearn_logistic": sk_results,
    }
)

comparison_df

### Discussion of Differences

Custom and sklearn results are often close when both models use the same feature set and scaling pipeline. Small differences can still appear due to implementation details such as optimization defaults, numerical tolerances, and stopping behavior. If differences are larger, they can also indicate sensitivity to class imbalance or feature preprocessing choices.